In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from copy import deepcopy
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
env, _ = setup_cart_pole_env()

In [ ]:
def featurize(obs):
    feat_obs = jnp.stack(
        [obs[..., 0], obs[..., 1], jnp.sin(obs[..., 2] * jnp.pi), jnp.cos(obs[..., 2] * jnp.pi), obs[..., 3]],
        axis=-1,
    )
    return feat_obs

In [ ]:
def simulate_ahead_with_env(env, env_properties, init_obs, init_state, actions):
    def body_fun(carry, action):
        obs, state = carry

        obs, state = env.step(state, action, env_properties)
        return (obs, state), obs

    (_, last_state), observations = jax.lax.scan(body_fun, (init_obs, init_state), actions)
    observations = jnp.concatenate([init_obs[None, :], observations], axis=0)

    return observations

In [ ]:
from dmpe.models.model_training import precompute_starting_points, load_single_batch

In [ ]:
key = jax.random.PRNGKey(3)
key, action_key, loader_key = jax.random.split(key, 3)

obs, state = env.reset(env.env_properties)
actions = jax.random.uniform(
    key=action_key,
    shape=(1000, 1),
    minval=-1.0,
    maxval=1.0,
)
observations = simulate_ahead_with_env(env, env.env_properties, obs, state, actions)

In [ ]:
sequence_length = 5
training_batch_size = 3

starting_points, loader_key = precompute_starting_points(
    n_train_steps=10,
    k=observations.shape[0],
    sequence_length=sequence_length,
    training_batch_size=training_batch_size,
    loader_key=loader_key,
)
batched_observations, batched_actions = load_single_batch(
    observations,
    actions,
    starting_points[0, ...],
    sequence_length,
    # obs_dim=observations.shape[-1],
    # act_dim=actions.shape[-1],
)

In [ ]:
starting_points.shape

In [ ]:
batched_observations.shape

In [ ]:
batched_actions.shape

In [ ]:
init_obs = batched_observations[:, 0]
init_state = eqx.filter_vmap(env.generate_state_from_observation, in_axes=(0, None))(init_obs, env.env_properties)

pred_observations = eqx.filter_vmap(simulate_ahead_with_env, in_axes=(None, None, 0, 0, 0))(
    env,
    env.env_properties,
    init_obs,
    init_state,
    batched_actions
)

for i in range(4):
    plt.plot(batched_observations[0, :20, i])
    plt.plot(pred_observations[0, :20, i], "--")
    plt.show()

for i in range(4):
    plt.plot(batched_observations[0, :20, i] - pred_observations[0, :20, i])
    plt.show()